# Module 9: RAG & LLM Applications
## Vector Databases: 04 - Managed Cloud Vector Databases & Pinecone

### Theoretical Overview
Pinecone provides fully managed, serverless vector search infrastructure with:
- Multi-tenant namespaces (`namespace="tenant_123"`).
- Metadata filtering on arbitrary JSON schemas.
- Zero-management autoscaling from thousands to billions of vectors.

In [ ]:
# Pinecone API interface mockup
class MockPineconeIndex:
    def __init__(self, name: str, dimension: int):
        self.name = name
        self.dimension = dimension
        self.namespaces = {}
        
    def upsert(self, vectors, namespace="default"):
        if namespace not in self.namespaces:
            self.namespaces[namespace] = {}
        for vec_id, values, metadata in vectors:
            self.namespaces[namespace][vec_id] = {"values": values, "metadata": metadata}
            
    def query(self, vector, top_k=2, namespace="default", filter=None):
        ns = self.namespaces.get(namespace, {})
        results = []
        q = np.array(vector)
        for vid, item in ns.items():
            if filter and not all(item["metadata"].get(k) == v for k, v in filter.items()):
                continue
            u = np.array(item["values"])
            score = float(np.dot(q, u) / (np.linalg.norm(q) * np.linalg.norm(u) + 1e-9))
            results.append({"id": vid, "score": score, "metadata": item["metadata"]})
        results.sort(key=lambda x: x["score"], reverse=True)
        return {"matches": results[:top_k], "namespace": namespace}

index = MockPineconeIndex(name="production-rag", dimension=4)
index.upsert(
    vectors=[
        ("vec_1", [0.1, 0.9, 0.3, 0.4], {"genre": "scifi", "year": 2024}),
        ("vec_2", [0.8, 0.1, 0.4, 0.2], {"genre": "finance", "year": 2023}),
    ],
    namespace="enterprise_docs"
)
res = index.query(vector=[0.12, 0.88, 0.31, 0.39], top_k=1, namespace="enterprise_docs", filter={"genre": "scifi"})
print("Pinecone Query Response:", res)